# **연수구 지방세 징수 데이터 분석 (NanumBarunGothicBold 적용)**
 
**같은 경로의 `NanumBarunGothicBold.ttf`를 강제 로드해서 한글 깨짐 0%**로 만든다.  
그리고 흐름은 딱 4단계:

> 로드 → 정제 → 분석 → 시각화 → 저장

---

## 핵심 설계

- 폰트: `NanumBarunGothicBold.ttf` (현재 실행 경로 기준)
- 단위: **십억 원**으로 스케일링 (가독성 ↑)
- 그래프:
  - 세목별 금액 비교 (부과 / 수납 / 미수납)
  - 징수율
  - 부과금액 비중 (파이)
  - 미수납 TOP

---

## 실행 코드

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")

# =========================
# 1) 경로 설정
# =========================
BASE_DIR = os.getcwd()
CSV_PATH = os.path.join(BASE_DIR, "Incheon Metropolitan City Yeonsu-gu_Local Tax Collection Status_20231231.csv")
FONT_PATH = os.path.join(BASE_DIR, "NanumBarunGothicBold.ttf")
OUTPUT_DIR = os.path.join(BASE_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 2) 한글 폰트 적용 (핵심)
# =========================
font_manager.fontManager.addfont(FONT_PATH)
font_name = font_manager.FontProperties(fname=FONT_PATH).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print(f"[INFO] Font Loaded: {font_name}")

# =========================
# 3) CSV 로드
# =========================
def load_csv(path):
    for enc in ["euc-kr", "cp949", "utf-8-sig", "utf-8"]:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[INFO] Encoding: {enc}")
            return df
        except:
            continue
    raise Exception("CSV 인코딩 실패")

df = load_csv(CSV_PATH)

# =========================
# 4) 데이터 정제
# =========================
df.columns = df.columns.str.strip()

num_cols = ["부과금액", "수납금액", "환급금액", "결손금액", "미수납 금액", "징수율"]

for col in num_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", "")
        .str.replace("-", "0")
    )
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["세목명"] = df["세목명"].astype(str).str.strip()

# =========================
# 5) 파생 변수
# =========================
df["부과금액_십억"] = df["부과금액"] / 1_000_000_000
df["수납금액_십억"] = df["수납금액"] / 1_000_000_000
df["미수납_십억"] = df["미수납 금액"] / 1_000_000_000

df = df.sort_values("부과금액", ascending=False)

# =========================
# 6) 핵심 지표
# =========================
total_assigned = df["부과금액"].sum()
total_collected = df["수납금액"].sum()
total_unpaid = df["미수납 금액"].sum()

collection_rate = (total_collected / total_assigned) * 100

print("\n===== 핵심 KPI =====")
print(f"총 부과금액: {total_assigned:,.0f}")
print(f"총 수납금액: {total_collected:,.0f}")
print(f"총 미수납 : {total_unpaid:,.0f}")
print(f"징수율     : {collection_rate:.2f}%")

# =========================
# 7) 시각화 함수
# =========================
def save(name):
    path = os.path.join(OUTPUT_DIR, name)
    plt.tight_layout()
    plt.savefig(path)
    print(f"[SAVED] {path}")

# =========================
# 8) 그래프 1: 금액 비교
# =========================
plt.figure(figsize=(14, 8))

x = np.arange(len(df))
w = 0.25

plt.bar(x - w, df["부과금액_십억"], width=w, label="부과")
plt.bar(x, df["수납금액_십억"], width=w, label="수납")
plt.bar(x + w, df["미수납_십억"], width=w, label="미수납")

plt.xticks(x, df["세목명"], rotation=45, ha="right")
plt.ylabel("십억 원")
plt.title("세목별 금액 비교")
plt.legend()
plt.grid(axis="y", alpha=0.3)

save("01_compare.png")
plt.show()

# =========================
# 9) 그래프 2: 징수율
# =========================
plt.figure(figsize=(14, 7))

bars = plt.bar(df["세목명"], df["징수율"])

for b, v in zip(bars, df["징수율"]):
    plt.text(b.get_x() + b.get_width()/2, v + 1, f"{v:.0f}%", ha="center")

plt.xticks(rotation=45, ha="right")
plt.ylabel("%")
plt.title("세목별 징수율")
plt.grid(axis="y", alpha=0.3)

save("02_rate.png")
plt.show()

# =========================
# 10) 그래프 3: 비중 (파이)
# =========================
plt.figure(figsize=(10, 10))

plt.pie(
    df["부과금액"],
    labels=df["세목명"],
    autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
    startangle=90
)

plt.title("부과금액 비중")

save("03_pie.png")
plt.show()

# =========================
# 11) 그래프 4: 미수납 TOP
# =========================
top = df.sort_values("미수납 금액", ascending=False).head(7)

plt.figure(figsize=(12, 6))
plt.barh(top["세목명"], top["미수납_십억"])

plt.xlabel("십억 원")
plt.title("미수납 TOP 7")
plt.grid(axis="x", alpha=0.3)

save("04_unpaid.png")
plt.show()

# =========================
# 12) CSV 저장
# =========================
df.to_csv(os.path.join(OUTPUT_DIR, "final.csv"), index=False, encoding="utf-8-sig")

print("\n[완료] 분석 + 시각화 + 저장 끝.")